[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.6 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads


    def causual_attn(self, x):
      # x = [B, S, D]
      B, S, D = x.shape
      assert D == self.d_model

      lnx = self.ln1(x)
      q = self.W_q(lnx)
      k = self.W_k(lnx)
      v = self.W_v(lnx)

      q = q.view(B, S, self.num_heads, self.d_head).transpose(1, 2)
      k = k.view(B, S, self.num_heads, self.d_head).transpose(1, 2)
      v = v.view(B, S, self.num_heads, self.d_head).transpose(1, 2) # [B, H, S, Dh]

      attn_score_no_softmax = q @ k.mT / math.sqrt(self.d_head)
      mask = torch.triu(torch.ones((S, S), dtype=torch.bool), diagonal=1)
      attn_score_no_softmax = torch.masked_fill(attn_score_no_softmax, mask, -math.inf)

      O = torch.softmax(attn_score_no_softmax, dim=-1) @ v # [B, H, S, Dh]
      O = O.transpose(1, 2).contiguous()
      O = O.view(B, S, self.d_model)
      return self.W_o(O)


    def forward(self, x):
      x = x + self.causual_attn(x)
      x = x + self.mlp(self.ln2(x))
      return x

In [7]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49728


In [8]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (6.5ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (0.9ms)
  ✅ [3/5] MLP has 4x expansion with GELU (1.1ms)
  ✅ [4/5] Causal masking — future doesn't affect past (4.0ms)
  ✅ [5/5] Gradient flow to all parameters (4.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (16.6ms total)
  Progress saved. Run status() to see your dashboard.

